# Empirical Complexity Analysis: STCON

This notebook analyzes the **Time Complexity** and **Space (Memory) Complexity** of the Sublinear STCON algorithm compared to standard Breadth-First Search (BFS).

### Theoretical Claims
1. **BFS Space**: $O(N)$ - Scales linearly as node count increases.
2. **STCON Space**: $O(N / 2^{\sqrt{\log N}})$ - Scales sublinearly, saving massive memory on huge graphs.
3. **Time Tradeoff**: STCON sacrifices time (exponentially higher recursive calculations) to save space.

In [ ]:
import time
import tracemalloc
import math
import random
import matplotlib.pyplot as plt

def generate_graph(n, edge_density=2):
    edges = [(i, i) for i in range(n)] + [(i, i+1) for i in range(n-1)]
    for _ in range(n * edge_density):
        edges.append((random.randint(0, n-1), random.randint(0, n-1)))
    adj_list = {i: set() for i in range(n)}
    for u, v in edges:
        adj_list[u].add(v)
    return adj_list


In [ ]:
# Algorithms
def standard_bfs(n, adj_list, start, target):
    visited = set([start])
    queue = [start]
    while queue:
        curr = queue.pop(0)
        if curr == target: return True
        for neighbor in adj_list[curr]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return False

def sublinear_stcon(n, adj_list, s, t):
    if n <= 1: return s == t
    log_n = max(1, math.log2(n))
    sqrt_log_n = math.sqrt(log_n)
    r = max(1, math.ceil(sqrt_log_n))
    k = max(1, math.ceil(2 ** sqrt_log_n))
    L = 2
    lambd = L ** r
    levels_to_check = math.floor(n / lambd) + 1

    def SPR(r_curr, ds, dt, Vs):
        Vt = set()
        if r_curr == 0:
            for u in Vs:
                if u % k == ds:
                    for v in adj_list[u]:
                        if v % k == dt:
                            Vt.add(v)
        else:
            for i in range(k ** (L - 1)):
                num = i
                digits = []
                for _ in range(L - 1):
                    digits.append(num % k)
                    num //= k
                sequence = [ds] + digits + [dt]
                current_V = set(Vs)
                for step in range(1, L + 1):
                    current_V = SPR(r_curr - 1, sequence[step - 1], sequence[step], current_V)
                Vt.update(current_V)
        return Vt

    for j in range(lambd):
        S = set([s])
        valid_j = True
        for level_idx in range(1, levels_to_check + 1):
            S_prime = set()
            for i1 in range(k):
                P = set()
                for i2 in range(k):
                    Q = set([v for v in S if v % k == i2])
                    if Q:
                        P.update(SPR(r, i2, i1, Q))
                if len(S) + len(S_prime.union(P)) > math.ceil(n / lambd) + 2:
                    valid_j = False; break
                S_prime.update(P)
            if not valid_j: break
            S.update(S_prime)
            if t in S: return True
        if valid_j and t in S: return True
    return False

In [ ]:
# Run Benchmark
graph_sizes = [10, 20, 50, 100, 150]
bfs_time, bfs_space = [], []
stcon_time, stcon_space = [], []

for n in graph_sizes:
    adj = generate_graph(n)
    
    tracemalloc.start()
    t0 = time.time()
    standard_bfs(n, adj, 0, n-1)
    t1 = time.time()
    _, p1 = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    bfs_time.append(t1 - t0)
    bfs_space.append(p1 / 1024)

    tracemalloc.start()
    t0 = time.time()
    sublinear_stcon(n, adj, 0, n-1)
    t1 = time.time()
    _, p2 = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    stcon_time.append(t1 - t0)
    stcon_space.append(p2 / 1024)

# Plotting Space Complexity
plt.figure(figsize=(10, 5))
plt.plot(graph_sizes, bfs_space, label='Standard BFS Space (Linear)', marker='o', color='red')
plt.plot(graph_sizes, stcon_space, label='STCON Space (Sublinear)', marker='o', color='blue')
plt.xlabel('Number of Nodes (N)')
plt.ylabel('Peak Memory (KB)')
plt.title('Space Complexity: BFS vs STCON')
plt.legend()
plt.grid(True)
plt.show()

# Plotting Time Complexity
plt.figure(figsize=(10, 5))
plt.plot(graph_sizes, bfs_time, label='Standard BFS Time', marker='o', color='red')
plt.plot(graph_sizes, stcon_time, label='STCON Time', marker='o', color='blue')
plt.xlabel('Number of Nodes (N)')
plt.ylabel('Execution Time (s)')
plt.title('Time Trade-off: BFS vs STCON')
plt.legend()
plt.grid(True)
plt.show()